# 🔬 PEFT A2Z — Autoresearch Reproduction

**Paper:** Prottasha et al. (2025), *PEFT A2Z: Parameter-Efficient Fine-Tuning Survey for Large Language and Vision Models*, arXiv:2504.14117v1

**What this notebook reproduces:** Table 1 (Section 6.1) — GLUE benchmark comparison of LoRA, BitFit, and Prefix Tuning on RoBERTa-Base.

**Tasks:** SST-2 (sentiment) and RTE (textual entailment)

---
### ⚠️ Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Then run cells top to bottom with **Runtime → Run all**

### ⏱️ Estimated time per experiment on T4:
| Method | Task | Time |
|--------|------|------|
| BitFit | RTE | ~5 min |
| LoRA | RTE | ~8 min |
| Prefix | RTE | ~10 min |
| BitFit | SST-2 | ~20 min |
| LoRA | SST-2 | ~30 min |
| Prefix | SST-2 | ~35 min |

**Tip:** Run RTE experiments first — they're fast and let you verify your results quickly.

## Cell 1 — Install dependencies

In [1]:
# Install required packages
!pip install -q transformers peft datasets scikit-learn accelerate
print('✅ Dependencies installed')

✅ Dependencies installed


## Cell 2 — Imports and configuration

In [2]:
import os
import csv
import time
import numpy as np
import torch
from datetime import datetime
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    PrefixTuningConfig,
    TaskType,
    get_peft_model,
)
from sklearn.metrics import accuracy_score

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

# ── Paper reference values from Table 1 ──
PAPER_RESULTS = {
    'lora':   {'sst2': 93.31, 'rte': 74.92},
    'bitfit': {'sst2': 93.12, 'rte': 77.98},
    'prefix': {'sst2': 93.81, 'rte': 54.51},
    'full':   {'sst2': 92.89, 'rte': 72.43},
}
PAPER_PARAMS = {'lora': 0.89, 'bitfit': 0.083, 'prefix': 0.96, 'full': 124.6}

# ── Task configuration ──
TASK_CONFIG = {
    'sst2': {
        'subset': 'sst2',
        'text_col': 'sentence',
        'num_labels': 2,
        'max_length': 128,
        'epochs': 5,
        'lr_lora': 3e-4,
        'lr_other': 1e-3,
    },
    'rte': {
        'subset': 'rte',
        'text_col': ['sentence1', 'sentence2'],
        'num_labels': 2,
        'max_length': 256,
        'epochs': 10,
        'lr_lora': 3e-4,
        'lr_other': 1e-3,
    },
}

MODEL_NAME = 'roberta-base'
RESULTS_FILE = 'glue_results.csv'

print('\n✅ Configuration ready')

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB

✅ Configuration ready


## Cell 3 — Helper functions

In [3]:
def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable / 1e6, total / 1e6


def apply_lora(model):
    """LoRA: Low-Rank Adaptation — Hu et al. (2021)"""
    config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=['query', 'value'],
        bias='none',
    )
    return get_peft_model(model, config)


def apply_bitfit(model):
    """BitFit: bias-only fine-tuning — Zaken et al. (2021)"""
    for name, param in model.named_parameters():
        if 'bias' not in name:
            param.requires_grad = False
    return model


def apply_prefix(model):
    """Prefix Tuning: learnable prefix tokens — Li & Liang (2021)"""
    config = PrefixTuningConfig(
        task_type=TaskType.SEQ_CLS,
        num_virtual_tokens=20,
        encoder_hidden_size=768,
    )
    return get_peft_model(model, config)


def tokenize_dataset(task_name, tokenizer):
    cfg = TASK_CONFIG[task_name]
    dataset = load_dataset('glue', cfg['subset'])

    if isinstance(cfg['text_col'], list):
        def tokenize_fn(examples):
            return tokenizer(
                examples[cfg['text_col'][0]],
                examples[cfg['text_col'][1]],
                truncation=True, max_length=cfg['max_length']
            )
    else:
        def tokenize_fn(examples):
            return tokenizer(
                examples[cfg['text_col']],
                truncation=True, max_length=cfg['max_length']
            )

    tokenized = dataset.map(tokenize_fn, batched=True)
    tokenized = tokenized.rename_column('label', 'labels')
    keep_cols = ['input_ids', 'attention_mask', 'labels']
    tokenized = tokenized.remove_columns(
        [c for c in tokenized['train'].column_names if c not in keep_cols]
    )
    tokenized.set_format('torch')
    return tokenized


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, predictions)}


def save_result(method, task, accuracy, trainable_m, elapsed_min, seed=42):
    paper_acc = PAPER_RESULTS[method][task]
    diff = accuracy - paper_acc
    fieldnames = ['timestamp', 'method', 'task', 'accuracy_ours',
                  'accuracy_paper', 'diff', 'trainable_params_M',
                  'paper_params_M', 'training_time_min', 'seed']
    file_exists = os.path.isfile(RESULTS_FILE)
    with open(RESULTS_FILE, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M'),
            'method': method, 'task': task,
            'accuracy_ours': round(accuracy, 2),
            'accuracy_paper': paper_acc,
            'diff': round(diff, 2),
            'trainable_params_M': round(trainable_m, 3),
            'paper_params_M': PAPER_PARAMS[method],
            'training_time_min': round(elapsed_min, 1),
            'seed': seed,
        })


def print_summary():
    if not os.path.isfile(RESULTS_FILE):
        print('No results yet. Run an experiment first.')
        return
    print(f"\n{'='*65}")
    print('RESULTS: Our Reproduction vs. Paper (Table 1, RoBERTa-Base)')
    print(f"{'='*65}")
    print(f"{'Method':<10} {'Task':<7} {'Ours':>8} {'Paper':>8} {'Diff':>7} {'Params(M)':>10}")
    print('-'*65)
    with open(RESULTS_FILE) as f:
        for row in csv.DictReader(f):
            print(f"{row['method']:<10} {row['task']:<7} "
                  f"{row['accuracy_ours']:>8} {row['accuracy_paper']:>8} "
                  f"{float(row['diff']):>+7.2f} {row['trainable_params_M']:>10}")
    print('-'*65)
    print(f"Full FT (paper): SST-2={PAPER_RESULTS['full']['sst2']}%  "
          f"RTE={PAPER_RESULTS['full']['rte']}%  Params={PAPER_PARAMS['full']}M")


print('✅ Helper functions defined')

✅ Helper functions defined


## Cell 4 — Main experiment runner

In [4]:
def run_experiment(method, task, seed=42):
    """
    Load RoBERTa-Base, apply a PEFT method, train on a GLUE task,
    evaluate on the validation set, and save results.

    Args:
        method: 'lora', 'bitfit', or 'prefix'
        task:   'sst2' or 'rte'
        seed:   random seed for reproducibility
    """
    print(f"\n{'='*55}")
    print(f"  {method.upper()} on {task.upper()}")
    print(f"{'='*55}")

    torch.manual_seed(seed)
    cfg = TASK_CONFIG[task]

    # ── Load model and tokenizer ──
    print(f'Loading {MODEL_NAME}...')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=cfg['num_labels']
    )

    # ── Apply PEFT ──
    print(f'Applying {method}...')
    if method == 'lora':
        model = apply_lora(model)
        lr = cfg['lr_lora']
    elif method == 'bitfit':
        model = apply_bitfit(model)
        lr = cfg['lr_other']
    elif method == 'prefix':
        model = apply_prefix(model)
        lr = cfg['lr_other']
    else:
        raise ValueError(f"Unknown method: {method}")

    trainable_m, total_m = count_trainable_params(model)
    pct = trainable_m / total_m * 100
    print(f'Trainable: {trainable_m:.3f}M / {total_m:.1f}M ({pct:.2f}%)')
    print(f'Paper reports: {PAPER_PARAMS[method]}M trainable params')

    # ── Tokenize dataset ──
    print(f'Loading {task} dataset...')
    tokenized = tokenize_dataset(task, tokenizer)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # ── Training arguments ──
    training_args = TrainingArguments(
        output_dir=f'./output_{method}_{task}',
        num_train_epochs=cfg['epochs'],
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        logging_steps=50,
        seed=seed,
        report_to='none',
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['validation'],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # ── Train ──
    print(f'\nTraining...')
    start = time.time()
    trainer.train()
    elapsed_min = (time.time() - start) / 60

    # ── Evaluate ──
    eval_results = trainer.evaluate()
    accuracy = eval_results['eval_accuracy'] * 100
    paper_acc = PAPER_RESULTS[method][task]

    print(f"\n{'─'*40}")
    print(f"  Our result  : {accuracy:.2f}%")
    print(f"  Paper result: {paper_acc:.2f}%")
    print(f"  Difference  : {accuracy - paper_acc:+.2f}%")
    print(f"  Time        : {elapsed_min:.1f} min")
    print(f"{'─'*40}")

    save_result(method, task, accuracy, trainable_m, elapsed_min, seed)
    print(f'Results saved to {RESULTS_FILE}')
    return accuracy


print('✅ run_experiment() ready')

✅ run_experiment() ready


## Cell 5 — Run: BitFit on RTE (fastest, ~5 min)
Start here to verify everything works before running longer experiments.

In [5]:
run_experiment('bitfit', 'rte')


  BITFIT on RTE
Loading roberta-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


────────────────────────────────────────
  Our result  : 63.18%
  Paper result: 77.98%
  Difference  : -14.80%
  Time        : 3.5 min
────────────────────────────────────────
Results saved to glue_results.csv


63.1768953068592

## Cell 6 — Run: LoRA on RTE (~8 min)

In [7]:
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.7 MB/s eta 0:00:00


In [8]:
run_experiment('lora', 'rte')


  LORA on RTE
Loading roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Applying lora...
Trainable: 0.887M / 125.5M (0.71%)
Paper reports: 0.89M trainable params
Loading rte dataset...


Map:   0%|          | 0/277 [00:00<?, ? examples/s]


Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.699009,0.715898,0.472924
2,0.659772,0.610294,0.667870
3,0.611857,0.614222,0.682310
4,0.562645,0.551426,0.711191
5,0.519296,0.577744,0.722022
6,0.465315,0.545936,0.725632
7,0.470486,0.573641,0.725632
8,0.418077,0.571594,0.718412
9,0.409778,0.602321,0.711191
10,0.372821,0.596961,0.722022



────────────────────────────────────────
  Our result  : 72.56%
  Paper result: 74.92%
  Difference  : -2.36%
  Time        : 3.5 min
────────────────────────────────────────
Results saved to glue_results.csv


72.56317689530685

## Cell 7 — Run: Prefix Tuning on RTE (~10 min)

In [9]:
run_experiment('prefix', 'rte')


  PREFIX on RTE
Loading roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Applying prefix...
Trainable: 0.961M / 125.6M (0.76%)
Paper reports: 0.96M trainable params
Loading rte dataset...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]


Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.736242,0.679365,0.541516
2,0.692904,0.701754,0.509025
3,0.686510,0.686008,0.530686
4,0.687384,0.676738,0.602888
5,0.670513,0.691086,0.530686
6,0.683962,0.671258,0.620939
7,0.677451,0.665738,0.617329
8,0.669009,0.675046,0.588448
9,0.673639,0.667546,0.624549
10,0.663618,0.666837,0.613718



────────────────────────────────────────
  Our result  : 62.45%
  Paper result: 54.51%
  Difference  : +7.94%
  Time        : 3.0 min
────────────────────────────────────────
Results saved to glue_results.csv


62.454873646209386

## Cell 8 — Run: BitFit on SST-2 (~20 min)

In [10]:
run_experiment('bitfit', 'sst2')


  BITFIT on SST2
Loading roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Applying bitfit...
Trainable: 0.103M / 124.6M (0.08%)
Paper reports: 0.083M trainable params
Loading sst2 dataset...


sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


────────────────────────────────────────
  Our result  : 93.46%
  Paper result: 93.12%
  Difference  : +0.34%
  Time        : 10.4 min
────────────────────────────────────────
Results saved to glue_results.csv


93.46330275229357

## Cell 9 — Run: LoRA on SST-2 (~30 min)

In [11]:
run_experiment('lora', 'sst2')


  LORA on SST2
Loading roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Applying lora...
Trainable: 0.887M / 125.5M (0.71%)
Paper reports: 0.89M trainable params
Loading sst2 dataset...


Map:   0%|          | 0/872 [00:00<?, ? examples/s]


Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.217798,0.206705,0.930046
2,0.190475,0.206692,0.933486
3,0.176699,0.198002,0.934633
4,0.159788,0.219972,0.933486
5,0.145874,0.215366,0.936927



────────────────────────────────────────
  Our result  : 93.69%
  Paper result: 93.31%
  Difference  : +0.38%
  Time        : 10.7 min
────────────────────────────────────────
Results saved to glue_results.csv


93.69266055045871

## Cell 10 — Run: Prefix Tuning on SST-2 (~35 min)

In [12]:
run_experiment('prefix', 'sst2')


  PREFIX on SST2
Loading roberta-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Applying prefix...
Trainable: 0.961M / 125.6M (0.76%)
Paper reports: 0.96M trainable params
Loading sst2 dataset...


Map:   0%|          | 0/1821 [00:00<?, ? examples/s]


Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.395259,0.289515,0.878440
2,0.295305,0.251520,0.901376
3,0.277751,0.231130,0.913991
4,0.281805,0.231912,0.915138
5,0.268398,0.233032,0.911697



────────────────────────────────────────
  Our result  : 91.51%
  Paper result: 93.81%
  Difference  : -2.30%
  Time        : 10.1 min
────────────────────────────────────────
Results saved to glue_results.csv


91.5137614678899

## Cell 11 — Print final results table

In [13]:
print_summary()


RESULTS: Our Reproduction vs. Paper (Table 1, RoBERTa-Base)
Method     Task        Ours    Paper    Diff  Params(M)
-----------------------------------------------------------------
bitfit     rte        63.18    77.98  -14.80      0.103
lora       rte        72.56    74.92   -2.36      0.887
prefix     rte        62.45    54.51   +7.94      0.961
bitfit     sst2       93.46    93.12   +0.34      0.103
lora       sst2       93.69    93.31   +0.38      0.887
prefix     sst2       91.51    93.81   -2.30      0.961
-----------------------------------------------------------------
Full FT (paper): SST-2=92.89%  RTE=72.43%  Params=124.6M


## Cell 12 — Download your results CSV
This downloads `glue_results.csv` to your local machine so you can add the numbers to your GitHub README.

In [14]:
from google.colab import files
if os.path.isfile(RESULTS_FILE):
    files.download(RESULTS_FILE)
    print(f'✅ Downloaded {RESULTS_FILE}')
else:
    print('No results file yet — run at least one experiment first.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded glue_results.csv
